In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from matplotlib.colors import TwoSlopeNorm
import logomaker
import itertools
from scipy.stats import spearmanr
from adjustText import adjust_text

In [ ]:
# params / paths
species = "Homo_sapiens"
species_space = "Homo sapiens"
rng = np.random.default_rng(42)
SEED = 42

subset_str = 'Alu' 
window_size = 400
window_left = window_size // 2
window_right = window_size // 2
window_abbr = f"{window_size/1000:.1f}"

print(f"window_size: {window_size}")
print(f"window_abbr: {window_abbr}")

## Data pre-processing

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

data = pd.read_csv(f"{species}/{species}_balanced_data_{window_abbr}kb_annotated_{subset_str}.csv")
print(f"data.shape: {data.shape}")
print(data['chrom'].value_counts())
print(data.head())

feature_cols = ['chrom', 'sequence', 'label', 'sv_type', 'sv_len', 'pos', 'end',
       'left_start', 'left_end', 'right_start', 'right_end', 'num_del',
       'num_ins', 'num_inv', 'num_smalldel', 'num_smallins', 'gc_content',
       'full_snp_count', 'distance_to_exon', 'distance_to_Alu/SINE',
       'distance_to_L1/LINE', 'distance_to_L2/LINE', 
       'distance_to_Low_complexity/Low_complexity', 'distance_to_MIR/SINE', 'distance_to_Satellite/Satellite',
       'distance_to_Simple_repeat/Simple_repeat',
       'avg_phyloP_scores', 'avg_recomb_rate_full', 'avg_cCRE_full',
       'avg_DNase_score_full', 'avg_tf_score_full']

balanced_data = data[feature_cols].copy()
balanced_data['num_ins_del'] = balanced_data['num_ins'] + balanced_data['num_del']
balanced_data['num_small_indel'] = balanced_data['num_smallins'] + balanced_data['num_smalldel']
balanced_data.drop(['num_ins', 'num_del', 'num_smallins', 'num_smalldel', 'left_end', 'right_start'], inplace=True, axis=1)

balanced_data.rename({
    'num_small_indel': 'Number of small indels',
    'num_ins_del': 'Number of large insertions and deletions',
    'num_inv': 'Number of inversions',
    'gc_content': 'GC content',
    'distance_to_Alu/SINE': 'Distance to nearest Alu/SINE',
    'distance_to_L1/LINE': 'Distance to nearest L1/LINE',
    'distance_to_L2/LINE': 'Distance to nearest L2/LINE',
    'distance_to_MIR/SINE': 'Distance to nearest MIR/SINE',
    'distance_to_Simple_repeat/Simple_repeat': 'Distance to nearest simple repeat',
    'distance_to_exon': 'Distance to nearest exon',
    'distance_to_Satellite/Satellite': 'Distance to nearest satellite',
    'distance_to_Low_complexity/Low_complexity': 'Distance to nearest low complexity repeat',
    'avg_phyloP_scores': 'Average phyloP score',
    'full_snp_count': 'Number of SNPs',
    'avg_DNase_score_full': 'Average DNase accessibility score',
    'avg_cCRE_full': 'Average cCRE score',
    'avg_tf_score_full': 'Average transcription factor peak',
    'avg_recomb_rate_full': 'Average recombination rate'
}, axis=1, inplace=True)

print(len(balanced_data['sequence'].iloc[0]))

mapping = {'A': [1, 0, 0, 0], 'T': [0, 1, 0, 0], 'C': [0, 0, 1, 0], 'G': [0, 0, 0, 1], 'N': [0, 0, 0, 0]}

def one_hot_encode(seq, mapping):
    return np.array([mapping[ch] for ch in seq])

def get_middle(seq):
    center = len(seq) // 2
    start = max(center - 500, 0)
    end = start + 1000
    return seq[start:end]

X_categorical = np.array([one_hot_encode(seq, mapping) for seq in balanced_data['sequence']])
y = balanced_data['label'].values
indices = np.arange(len(y))

num_feature_cols = ['Number of large insertions and deletions',
       'Number of small indels', 'Number of inversions', 'GC content',
       'Number of SNPs', 'Distance to nearest exon', 'Distance to nearest Alu/SINE',
       'Distance to nearest L1/LINE', 'Distance to nearest L2/LINE', 
       'Distance to nearest low complexity repeat', 'Distance to nearest MIR/SINE', 'Distance to nearest satellite',
       'Distance to nearest simple repeat',
       'Average phyloP score', 'Average recombination rate', 'Average cCRE score',
       'Average DNase accessibility score', 'Average transcription factor peak']

X_numerical = balanced_data[num_feature_cols].astype(float).to_numpy()
print(X_categorical.shape, X_numerical.shape, y.shape)

X_train_val, X_test, y_train_val, y_test, X_num_train_val, X_num_test, train_val_idx, test_idx = train_test_split(
    X_categorical, y, X_numerical, indices, test_size=0.20, random_state=SEED, stratify=y
)

X_train, X_val, y_train, y_val, X_num_train, X_num_val, train_idx, val_idx = train_test_split(
    X_train_val, y_train_val, X_num_train_val, train_val_idx, test_size=0.25, random_state=SEED, stratify=y_train_val
)

len(test_idx), test_idx[:10]

## Loading CNN

In [ ]:
n = 2
num_layers = f"{n}"
print(f"n: {num_layers}")

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=11, stride=stride, padding=5) 
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=False)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=11, stride=stride, padding=5)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.downsample = None

    def forward(self, x):
        residual = x
        if self.downsample:
            residual = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        out += residual
        out = self.relu(out)
        return out

class SV_CNN(nn.Module):
    def __init__(self, num_cat_channels, num_classes):
        super(SV_CNN, self).__init__()
        self.initial = nn.Sequential(
            nn.Conv1d(num_cat_channels, 128, kernel_size=1, stride=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=False)
        )

        self.layers = nn.ModuleList()
        self.layers.append(ResidualBlock(128, 128))
        for _ in range(n):  
            self.layers.append(ResidualBlock(128, 128))

        self.final_conv = nn.Conv1d(128, 32, kernel_size=1)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        
        self.fc1 = nn.Linear(12800, 128)
        self.fc2 = nn.Linear(128, num_classes)
        

    def forward(self, x_cat):
        x_cat = self.initial(x_cat)
        for layer in self.layers:
            x_cat = layer(x_cat)
        x_cat = self.final_conv(x_cat)
        
        x_cat = x_cat.view(x_cat.size(0), -1)
        x_cat = self.fc1(x_cat)
        x_cat = self.relu(x_cat)
        x_cat = self.fc2(x_cat)
        
        x = torch.sigmoid(x_cat)
        return x

    def extract_features(self, x_cat):
        x_cat = self.initial(x_cat)
        for layer in self.layers:
            x_cat = layer(x_cat)
        x_cat = self.final_conv(x_cat)
        x_cat = x_cat.view(x_cat.size(0), -1)
        x_cat = self.fc1(x_cat)
        x_cat = self.relu(x_cat)
        return x_cat 

model = SV_CNN(4, 1)
if torch.cuda.is_available():
    model.to(device)
print(device)
print(model)

model.load_state_dict(torch.load(f"{species}/{species}_{window_abbr}kb_best_model_{num_layers}_{subset_str}.pt", map_location=torch.device(device)))
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Alu-associated SVs in test set

In [ ]:
test_df = pd.read_csv(f"{species}/{species}_{window_abbr}kb_test_df_{subset_str}.csv")
test_df["sequence"] = balanced_data["sequence"].iloc[test_idx].values
test_df["sv_len"] = balanced_data["sv_len"].iloc[test_idx].values

# Alu-associated positives (label == 1)
alu_test_df = test_df[test_df["True label"] == 1].reset_index(drop=True)
sv_lens = alu_test_df["sv_len"].values
counts, bin_edges = np.histogram(sv_lens, bins=10)

# Find peak bin
peak_idx = np.argmax(counts)
bin_start = bin_edges[peak_idx]
bin_end = bin_edges[peak_idx + 1]

alu_test_df = alu_test_df[
    (alu_test_df["sv_len"] >= bin_start) &
    (alu_test_df["sv_len"] < bin_end)
].reset_index(drop=True)

,Number of large insertions and deletions,Number of small indels,Number of inversions,GC content,Number of SNPs,Distance to nearest exon,Distance to nearest Alu/SINE,Distance to nearest L1/LINE,Distance to nearest L2/LINE,Distance to nearest low complexity repeat,...,Average recombination rate,Average cCRE score,Average DNase accessibility score,Average transcription factor peak,CNN predicted probability,RF probability,Ensemble probability,True label,sequence,sv_len
0,0.0,0.0,0.0,33.00,3.0,200.0,200.0,200.0,200.0,200.0,...,5.318060e-25,0.000000,0.0,0.000,0.898840,0.567,0.507,1,TAAAACTTGGCAAGAACCATAGATTTGCGATGTATTATTGGTGATA...,333
1,0.0,0.0,0.0,49.50,7.0,47.0,43.0,200.0,200.0,200.0,...,2.850880e-01,0.000000,0.0,0.000,0.311429,0.704,0.513,1,GGTGTGGTGGTGCATGCCTATAGTTCCAGCTACTCAGGAGGCTGAG...,248
2,0.0,2.0,0.0,33.00,2.0,200.0,200.0,200.0,200.0,200.0,...,4.358750e-01,0.000000,0.0,0.000,0.972241,0.514,0.574,1,GACAAGCTGCCTGTAAATATTTGTATCTTGCCTTTTAAAGGAGTGC...,346
3,0.0,1.0,0.0,60.75,2.0,200.0,192.0,200.0,200.0,200.0,...,8.997690e-01,0.000000,0.0,0.000,0.508828,0.217,0.256,1,GGTGAAGAGGGGATTCGAACCCTGAGCCATCTAAGGTCCAGGGCCC...,320
4,0.0,0.0,0.0,29.25,4.0,200.0,200.0,88.0,200.0,200.0,...,1.940378e-03,0.000000,0.0,0.000,0.111127,0.467,0.455,1,TTAGTATTCTATAAGTTTTCCAAGATTAATTAATTTGATCTAATTT...,332
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2050,0.0,0.0,0.0,33.00,4.0,200.0,200.0,200.0,200.0,200.0,...,2.766340e-03,0.000000,0.0,0.000,0.897454,0.664,0.576,1,TAATCCATTGCGTGCCAAGCTAATAGATTTATATTTCATCTGTCAA...,343
2051,0.0,1.0,0.0,32.50,2.0,200.0,200.0,200.0,200.0,200.0,...,8.498900e-02,0.000000,130.5,82.875,0.193840,0.572,0.440,1,TTTTTCCTAGTACTTCTGTGAATTTTTTTATATCAGAATGAATTTG...,321
2052,0.0,0.0,0.0,41.25,2.0,200.0,200.0,200.0,200.0,200.0,...,1.683810e-01,4.883180,1000.0,0.000,0.287183,0.244,0.242,1,ATTCAGACAACGAATGCTCAGAAGGCCAGAACCCCTGTCTCTGCAG...,333
2053,0.0,0.0,0.0,38.25,0.0,200.0,200.0,200.0,200.0,200.0,...,1.770550e+00,0.856388,78.5,0.000,0.984985,0.474,0.554,1,CAGAGGCTCTTGTCAAGTCTCTGCTGGCATTCCACTTGCTGTTGTC...,316


## In silico mutagenesis

In [ ]:
center = window_size // 2
positions_to_mutate = range(center - 30, center + 31)

BASES = ["A", "C", "G", "T"]

def one_hot_encode(seq):
    mapping = {"A":0, "C":1, "G":2, "T":3}
    arr = np.zeros((len(seq), 4), dtype=np.float32)
    for i, base in enumerate(seq):
        if base in mapping:
            arr[i, mapping[base]] = 1.0
    return arr

def predict_prob_batch(seqs, batch_size=1024):
    model.eval()
    probs = []

    with torch.no_grad():
        for i in range(0, len(seqs), batch_size):
            batch_seqs = seqs[i:i + batch_size]

            batch = np.stack([one_hot_encode(s) for s in batch_seqs])
            batch = torch.tensor(batch, dtype=torch.float32).permute(0, 2, 1).to(device)

            logits = model(batch)
            batch_probs = torch.sigmoid(logits).squeeze(-1)

            probs.append(batch_probs.cpu().numpy())

    return np.concatenate(probs)


def insilico_mutagenesis_fast(seq, BASES, predict_prob, window=30, batch_size=1024):
    L = len(seq)
    center = L // 2
    positions = np.arange(center - window, center + window + 1)
    wt_prob = predict_prob([seq])[0]

    # Generate all mutant sequences
    mutated_seqs = []
    for pos in positions:
        for base in BASES:
            s = list(seq)
            s[pos] = base
            mutated_seqs.append("".join(s))

    mutated_probs = predict_prob(mutated_seqs, batch_size=batch_size)
    mutated_probs = mutated_probs.reshape(len(positions), len(BASES))
    delta = mutated_probs - wt_prob

    # Normalize per position (subtract reference base)
    for i, pos in enumerate(positions):
        ref_idx = BASES.index(seq[pos])
        delta[i, :] -= delta[i, ref_idx]

    return delta

In [ ]:
window = 30
Lwin = 2 * window + 1
BASES = ["A", "C", "G", "T"]

avg_delta = np.zeros((Lwin, 4))
counts = np.zeros((Lwin, 4))

base_to_idx = {b: i for i, b in enumerate(BASES)}

for seq in alu_test_df["sequence"]:
    delta = insilico_mutagenesis_fast(
        seq,
        BASES,
        predict_prob_batch,
        window=window
    )

    center = len(seq) // 2

    for i in range(Lwin):
        ref_base = seq[center - window + i]
        ref_idx = base_to_idx[ref_base]

        for b in range(4):
            if b != ref_idx:
                avg_delta[i, b] += delta[i, b]
                counts[i, b] += 1

# Normalize by number of contributing mutations
avg_delta /= np.maximum(counts, 1)

In [ ]:
def plot_avg_heatmap(avg_delta):
    L = avg_delta.shape[0]
    center = L // 2
    start = center - 30
    end = center + 31

    region = avg_delta[start:end, :]
    vmax = np.nanmax(np.abs(region))
    norm = TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax)

    plt.figure(figsize=(14, 4))
    plt.imshow(region.T, aspect="auto", cmap="coolwarm", norm=norm)
    plt.yticks(range(4), BASES)
    plt.xticks(range(61), range(-30, 31))
    plt.xlabel("Position relative to breakpoint (bp)")
    plt.ylabel("Mutated base")
    plt.colorbar(label="Δ SV probability")
    plt.title(f"In-silico Mutagenesis Heatmap ±30bp - {subset_str}")
    plt.tight_layout()
    plt.savefig(f"{species}/{species}_{window_abbr}kb_ISM_heatmap_{subset_str}.pdf",
                dpi=300, bbox_inches='tight')
    plt.show()

plot_avg_heatmap(avg_delta)

In [ ]:
def make_logo_30bp(avg_delta, center=None, title="Motif Sensitivity (±30 bp)"):
    L = avg_delta.shape[0]
    if center is None:
        center = L // 2

    start = center - 30
    end = center + 31

    sub = avg_delta[start:end, :]
    df = pd.DataFrame(sub, columns=BASES)
    df.index = np.arange(-30, 31)

    plt.figure(figsize=(16, 4))
    logomaker.Logo(df, color_scheme="classic")

    plt.axhline(0, color="black", linewidth=1)
    plt.xlabel("Position relative to breakpoint")
    plt.ylabel("Δ predicted SV probability")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f"{species}/{species}_{window_abbr}kb_ISM_PWM_{subset_str}.pdf",
                format="pdf", dpi=300, bbox_inches='tight')
    plt.show()

make_logo_30bp(avg_delta, title=f"{subset_str} - ISM")

In [ ]:
def make_ism_logo_relative(avg_delta, window=30, center=None,
                           title="ISM Sensitivity Logo (relative magnitude)"):
    L = avg_delta.shape[0]
    if center is None:
        center = L // 2

    start = center - window
    end = center + window + 1
    sub = avg_delta[start:end, :]
    sub = np.clip(sub, 0, None)

    max_val = np.max(sub)
    if max_val > 0:
        sub = sub / max_val

    df = pd.DataFrame(sub, columns=BASES)
    df.index = np.arange(-window, window + 1)

    plt.figure(figsize=(16, 4))
    logomaker.Logo(
        df,
        color_scheme="classic",
        center_values=False
    )

    plt.xlabel("Position relative to breakpoint (bp)")
    plt.ylabel("Relative ISM magnitude")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(
        f"{species}/{species}_{window_abbr}kb_ISM_logo_relative_{subset_str}.pdf",
        format="pdf",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

make_ism_logo_relative(
    avg_delta,
    window=30,
    title=f"{subset_str} - ISM"
)

In [ ]:
pos_df = alu_test_df.copy()
neg_df = test_df[test_df["True label"] == 0].reset_index(drop=True)

L = len(pos_df["sequence"].iloc[0])
center = L // 2

start = center - 30
end = center + 31
rel_positions = np.arange(-30, 31)

def base_frequency_matrix(df):
    freq = np.zeros((len(rel_positions), 4))
    mapping = {"A":0, "C":1, "G":2, "T":3}

    for seq in df["sequence"]:
        window = seq[start:end]
        for i, base in enumerate(window):
            if base in mapping:
                freq[i, mapping[base]] += 1

    # convert counts to frequency
    freq = freq / len(df)
    return freq

pos_freq = base_frequency_matrix(pos_df)
neg_freq = base_frequency_matrix(neg_df)

neg_freq = np.clip(neg_freq, 1e-6, None)

# Compute enrichment ratio
enrichment = pos_freq / neg_freq 
freq_mat = base_frequency_matrix(alu_test_df)

# Convert to information content (bits)
eps = 1e-6
p = np.clip(freq_mat, eps, 1.0)

H = -np.sum(p * np.log2(p), axis=1)
H_max = np.log2(4) 
R = H_max - H
logo_bits = p * R[:, None]

logo_df = pd.DataFrame(logo_bits, columns=BASES)
logo_df.index = rel_positions

plt.figure(figsize=(10, 3))
logomaker.Logo(
    logo_df,
    color_scheme="classic"
)

plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Position relative to breakpoint (bp)")
plt.ylabel("Bits")
plt.title(f"{subset_str} - Empirical")
plt.tight_layout()

plt.savefig(
    f"{species}/{species}_{window_abbr}kb_empirical_logo_bits_10bp_{subset_str}.pdf",
    format="pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## Motif insertions at breakpoint

In [ ]:
def replace_kmer(seq, kmer, pos):
    """
    Replace sequence[pos : pos + len(kmer)] with kmer.
    Length of sequence is preserved.
    """
    return seq[:pos] + kmer + seq[pos + len(kmer):]

bg_df = test_df[test_df["True label"] == 0].reset_index(drop=True)

BASES = ["A", "C", "G", "T"]
kmers = ["".join(p) for p in itertools.product(BASES, repeat=4)]

results = []

for _, row in bg_df.iterrows():
    seq = row["sequence"]
    L = len(seq)
    mid = L // 2

    one_hot = one_hot_encode(seq).T
    input_tensor = torch.tensor(one_hot, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        original_prob = model(input_tensor).item()

    for kmer in kmers:
        # place kmer so that its 2nd base is at breakpoint
        pos = mid - 1
        modified_seq = replace_kmer(seq, kmer, pos)
        one_hot_mod = one_hot_encode(modified_seq).T
        input_tensor_mod = torch.tensor(one_hot_mod, dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            modified_prob = model(input_tensor_mod).item()

        results.append({
            "kmer": kmer,
            "delta": modified_prob - original_prob,
            "modified_prob": modified_prob,
            "original_prob": original_prob
        })

df = pd.DataFrame(results)
summary = (
    df.groupby("kmer")["delta"]
      .agg(["mean", "std", "count"])
      .sort_values("mean", ascending=False)
)

In [ ]:
top = summary.head(5)

plt.figure(figsize=(5, 5))
plt.bar(
    top.index,
    top["mean"],
    yerr=top["std"],
    capsize=4,
    color="#4C72B0",
    edgecolor="black",
    linewidth=0.6,
    alpha=0.9
)

plt.axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.6)
plt.ylabel("Δ probability", fontsize=12)
plt.xlabel("4-mer", fontsize=12)
plt.title("Top activating 4-mers at breakpoint", fontsize=14)
plt.xticks(rotation=90, fontsize=10)
plt.tight_layout()
plt.savefig(
    f"{species}/{species}_top_activating_kmers_breakpoint_{subset_str}.pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
k = 4

def extract_kmers(seq, k):
    return [seq[i:i+k] for i in range(len(seq) - k + 1)]

def collect_kmers(df, k=4, window=30):
    kmers = []

    for seq in df["sequence"]:
        L = len(seq)
        center = L // 2
        start = center - window
        end = center + window + 1

        region = seq[start:end]

        kmers.extend(extract_kmers(region, k))

    return kmers


pos_kmers = collect_kmers(
    test_df[test_df["True label"] == 1],
    k=k, window=10
)

neg_kmers = collect_kmers(
    test_df[test_df["True label"] == 0],
    k=k, window=10
)

pos_counts = Counter(pos_kmers)
neg_counts = Counter(neg_kmers)
all_kmers = set(pos_counts) | set(neg_counts)

results = []

for kmer in all_kmers:
    pos = pos_counts.get(kmer, 0)
    neg = neg_counts.get(kmer, 0)

    pos_freq = pos / sum(pos_counts.values())
    neg_freq = neg / sum(neg_counts.values())

    log2_fc = np.log2((pos_freq + 1e-6) / (neg_freq + 1e-6))

    results.append({
        "kmer": kmer,
        "log2FC": log2_fc,
        "pos_freq": pos_freq,
        "neg_freq": neg_freq
    })

df_kmer = pd.DataFrame(results).sort_values("log2FC", ascending=False)

results = []
total_pos = sum(pos_counts.values())
total_neg = sum(neg_counts.values())

for kmer in all_kmers:
    a = pos_counts.get(kmer, 0)
    b = neg_counts.get(kmer, 0)
    c = total_pos - a
    d = total_neg - b

    results.append({
        "kmer": kmer,
        "log2FC": np.log2((a + 1) / (b + 1))
    })

df_stats = pd.DataFrame(results)

In [ ]:
fc_thresh = 1.2
delta_thresh = 0.3

df_plot = summary.reset_index().merge(
    df_stats[["kmer", "log2FC"]],
    on="kmer",
    how="left"
)

rho, pval = spearmanr(df_plot["log2FC"], df_plot["mean"])

df_highlight = df_plot[
    (df_plot["log2FC"] > fc_thresh) |
    (df_plot["mean"] > delta_thresh)
]

df_other = df_plot.drop(df_highlight.index)

plt.figure(figsize=(6.5, 5))

plt.scatter(
    df_other["log2FC"],
    df_other["mean"],
    color="#b0b0b0",
    alpha=0.6,
    s=35,
    label="Other 4-mers"
)

plt.scatter(
    df_highlight["log2FC"],
    df_highlight["mean"],
    color="#c0392b",
    s=70,
    label="log2FC > 1.2 & Δ > 0.2"
)

texts = []
for _, row in df_highlight.iterrows():
    texts.append(
        plt.text(
            row["log2FC"],
            row["mean"],
            row["kmer"],
            fontsize=9,
            fontweight="bold",
            color="#c0392b",
            ha="left",
            va="bottom"
        )
    )

adjust_text(
    texts,
    arrowprops=dict(arrowstyle="-", color="gray", lw=0.5),
    expand_text=(1.1, 1.2),
    expand_points=(1.1, 1.2)
)

plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.axvline(0, color="black", linestyle="--", linewidth=1)

plt.text(
    0.02, 0.98,
    f"Spearman ρ = {rho:.3f}\np = {pval:.2e}",
    transform=plt.gca().transAxes,
    ha="left",
    va="top",
    fontsize=11,
    bbox=dict(facecolor="white", alpha=0.85, edgecolor="black")
)

plt.xlabel("log2 fold enrichment")
plt.ylabel("Δ probability (mean effect)")
plt.title("4-mer enrichment vs functional effect at breakpoint")

plt.tight_layout()
plt.savefig(
    f"{species}/{species}_kmer_enrichment_vs_effect_{subset_str}.pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.show()